# Moduł 7: Regresja liniowa i spadek gradientowy

To **fundament** uczenia sieci neuronowych — każda sieć uczy się przez optymalizację gradientową!

## Spis treści
1. Regresja liniowa — teoria
2. Funkcja kosztu (MSE)
3. Spadek gradientowy (Gradient Descent)
4. Implementacja w NumPy
5. Wprowadzenie do PyTorch
6. Regresja liniowa w PyTorch
7. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
sns_available = False
try:
    import seaborn as sns
    sns.set_style("whitegrid")
    sns_available = True
except:
    pass

## 1. Regresja liniowa — teoria

**Cel:** Znaleźć prostą (lub hiperpłaszczyznę) najlepiej dopasowaną do danych.

### Model — przypadek jednowymiarowy
$$\hat{y} = w \cdot x + b$$

- $w$ — waga (slope / nachylenie)
- $b$ — bias (intercept / wyraz wolny)
- $\hat{y}$ — predykcja

### Model — przypadek wielowymiarowy

Dla $d$ cech:

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b = \sum_{j=1}^{d} w_j x_j + b$$

W notacji macierzowej (dla $n$ próbek): $\hat{\mathbf{y}} = X\mathbf{w} + b$, gdzie $X \in \mathbb{R}^{n \times d}$.

### Rozwiązanie analityczne (Normal Equation)

Minimalizacja MSE ma rozwiązanie w postaci zamkniętej:

$$\mathbf{w}^* = (X^T X)^{-1} X^T \mathbf{y}$$

To **pseudoinwersja Moore'a-Penrose'a**. Złożoność: $O(nd^2 + d^3)$.

- **Zaleta**: dokładne rozwiązanie w jednym kroku
- **Wada**: $d^3$ — nie skaluje się dla dużych $d$, oraz $X^TX$ musi być odwracalna

### Cel uczenia — minimalizacja funkcji straty
Znaleźć $w$ i $b$, które **minimalizują błąd** między $\hat{y}$ a prawdziwym $y$.

### Inne funkcje straty dla regresji

| Nazwa | Wzór | Własności |
|-------|------|-----------|
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Gładka, wrażliwa na outliery |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Odporna na outliery, nie różniczkowalna w 0 |
| **Huber** | $\begin{cases}\frac{1}{2}(y-\hat{y})^2 & \text{jeśli } \|y-\hat{y}\| \leq \delta \\ \delta(\|y-\hat{y}\| - \frac{\delta}{2}) & \text{w.p.p.}\end{cases}$ | Kompromis MSE/MAE |

In [ ]:
# Generujeym dane: y = 3x + 7 + szum
np.random.seed(42)
X = np.random.uniform(-5, 5, 100)
y_true = 3 * X + 7 + np.random.normal(0, 2, 100)

plt.figure(figsize=(8, 5))
plt.scatter(X, y_true, alpha=0.6, label="Dane")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dane do regresji liniowej")
plt.legend()
plt.show()

## 2. Funkcja kosztu — MSE

**MSE (Mean Squared Error):**

$$\mathcal{L}(\mathbf{w}, b) = \text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n}\|\mathbf{y} - X\mathbf{w}\|_2^2$$

- Kwadrat kary za duże błędy → duże odchylenia "bolą" bardziej
- Zawsze $\geq 0$, minimum = 0 (idealne dopasowanie)
- **Funkcja wypukła** (convex) → gwarantowane minimum globalne
- Różniczkowalna wszędzie → łatwa optymalizacja gradientowa

### Krajobraz funkcji kosztu

Dla 2 parametrów ($w$, $b$) MSE tworzy **paraboloidę** — miskę z jednym minimum. Gradienty wskazują "w dół" miseczki.

### Regularyzacja — zapobieganie overfittingowi

$$\mathcal{L}_{\text{reg}} = \text{MSE} + \lambda \cdot \Omega(\mathbf{w})$$

| Regularyzacja | $\Omega(\mathbf{w})$ | Efekt | sklearn |
|--------------|---------------------|-------|---------|
| **L2 (Ridge)** | $\|\mathbf{w}\|_2^2 = \sum w_j^2$ | Zmniejsza wagi | `Ridge` |
| **L1 (Lasso)** | $\|\mathbf{w}\|_1 = \sum\|w_j\|$ | Zeruje wagi (feature selection) | `Lasso` |
| **Elastic Net** | $\alpha\|\mathbf{w}\|_1 + (1-\alpha)\|\mathbf{w}\|_2^2$ | Kompromis L1/L2 | `ElasticNet` |

In [ ]:
def predict(X, w, b):
    return w * X + b

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# Spróbujmy różne wartości w i zobaczmy MSE
w_values = np.linspace(-2, 8, 100)
b_fixed = 7
losses = [mse(y_true, predict(X, w, b_fixed)) for w in w_values]

plt.figure(figsize=(8, 5))
plt.plot(w_values, losses)
plt.xlabel("w (waga)")
plt.ylabel("MSE")
plt.title("Krajobraz funkcji kosztu (b=7)")
plt.axvline(x=3, color="red", linestyle="--", label="Optimum (w=3)")
plt.legend()
plt.show()

## 3. Spadek gradientowy (Gradient Descent)

**Idea:** Iteracyjnie aktualizuj parametry w kierunku **przeciwnym** do gradientu (pochodnej) funkcji kosztu.

### Reguły aktualizacji

$$w := w - \alpha \cdot \frac{\partial \text{MSE}}{\partial w}$$
$$b := b - \alpha \cdot \frac{\partial \text{MSE}}{\partial b}$$

gdzie $\alpha > 0$ to **learning rate** (tempo uczenia).

### Gradienty MSE

$$\frac{\partial \text{MSE}}{\partial w} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \cdot x_i$$

$$\frac{\partial \text{MSE}}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$

W notacji wektorowej: $\nabla_\mathbf{w} \mathcal{L} = -\frac{2}{n} X^T(\mathbf{y} - X\mathbf{w})$

### Warianty Gradient Descent

| Wariant | Rozmiar batcha | Zalety | Wady |
|---------|---------------|--------|------|
| **Batch GD** | $n$ (wszystkie) | Stabilna konwergencja | Wolne dla dużych $n$ |
| **SGD** | 1 | Szybkie iteracje | Niestabilne, szumne |
| **Mini-batch GD** | $B$ (np. 32, 64) | Kompromis | Wymaga tuning $B$ |

### Learning rate — wpływ na konwergencję

- **Za mały $\alpha$** → bardzo wolna zbieżność, ryzyko utknięcia w minimum lokalnym
- **Za duży $\alpha$** → oscylacje, dywergencja (loss rośnie!)
- **W sam raz** → szybka i stabilna zbieżność

### Zaawansowane optymalizatory

**Momentum** — dodaje "bezwładność" do aktualizacji:
$$v_t = \beta v_{t-1} + \nabla \mathcal{L}(\theta_t)$$
$$\theta_{t+1} = \theta_t - \alpha \cdot v_t$$

**RMSProp** — adaptacyjny learning rate per parametr:
$$s_t = \beta s_{t-1} + (1-\beta)(\nabla \mathcal{L})^2$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{s_t + \epsilon}} \nabla \mathcal{L}$$

**Adam** (Adaptive Moment Estimation) — **najpopularniejszy optymalizator**:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) \nabla\mathcal{L} \quad \text{(moment 1. rzędu — średnia)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) (\nabla\mathcal{L})^2 \quad \text{(moment 2. rzędu — wariancja)}$$

Korekta biasu:
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Aktualizacja:
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Domyślne hiperparametry: $\beta_1 = 0.9, \beta_2 = 0.999, \epsilon = 10^{-8}$

### Warunek zbieżności

Dla funkcji **silnie wypukłej** z ciągłym gradientem ($L$-Lipschitz):
$$\alpha < \frac{2}{L}$$

gwarantuje zbieżność, gdzie $L$ to stała Lipschitza gradientu.

### Learning Rate Scheduling

Zmiana $\alpha$ w trakcie treningu:
- **Step decay**: $\alpha_t = \alpha_0 \cdot \gamma^{\lfloor t/S \rfloor}$ (co $S$ epok mnóż przez $\gamma < 1$)
- **Cosine annealing**: $\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})(1 + \cos(\pi t / T))$
- **Warmup**: liniowo zwiększaj $\alpha$ przez kilka pierwszych epok

## 4. Implementacja w NumPy

In [ ]:
# Gradient Descent — implementacja od zera!

# Inicjalizacja losowa
w = np.random.randn()
b = np.random.randn()
lr = 0.01          # learning rate
epochs = 200       # ile iteracji
n = len(X)

history = {"epoch": [], "w": [], "b": [], "loss": []}

for epoch in range(epochs):
    # 1. Forward pass (predykcja)
    y_pred = w * X + b
    
    # 2. Oblicz loss
    loss = mse(y_true, y_pred)
    
    # 3. Oblicz gradienty
    dw = -(2/n) * np.sum((y_true - y_pred) * X)
    db = -(2/n) * np.sum(y_true - y_pred)
    
    # 4. Aktualizuj parametry
    w -= lr * dw
    b -= lr * db
    
    # Zapisz historię
    history["epoch"].append(epoch)
    history["w"].append(w)
    history["b"].append(b)
    history["loss"].append(loss)

print(f"Wynik: w = {w:.3f} (oczekiwane: 3), b = {b:.3f} (oczekiwane: 7)")
print(f"Końcowy MSE: {history['loss'][-1]:.4f}")

In [ ]:
# Wizualizacja: krzywa uczenia + dopasowanie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Krzywa uczenia
axes[0].plot(history["epoch"], history["loss"])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].set_title("Krzywa uczenia")

# Dopasowanie
x_line = np.linspace(-5, 5, 100)
axes[1].scatter(X, y_true, alpha=0.5, label="Dane")
axes[1].plot(x_line, w * x_line + b, color="red", linewidth=2,
             label=f"y = {w:.2f}x + {b:.2f}")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Dopasowana prosta")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Wprowadzenie do PyTorch

**PyTorch** to framework do deep learningu. Kluczowe koncepty:

### Tensor
Jak `numpy.ndarray`, ale:
- Może być na GPU (`tensor.to("cuda")`)
- Ma **autograd** — automatyczne różniczkowanie!
- Wspiera broadcasting (jak NumPy)

### Autograd — automatyczne różniczkowanie

PyTorch buduje **dynamiczny graf obliczeniowy** (computational graph). Każda operacja na tensorach z `requires_grad=True` jest zapisywana, a `backward()` oblicza gradienty metodą **łańcuchowej reguły** (chain rule):

$$\frac{\partial \mathcal{L}}{\partial \theta_j} = \frac{\partial \mathcal{L}}{\partial o_n} \cdot \frac{\partial o_n}{\partial o_{n-1}} \cdots \frac{\partial o_{k+1}}{\partial \theta_j}$$

### Trening w PyTorch — schemat

```
for epoch:
    y_pred = model(X)           # Forward pass
    loss = criterion(y_pred, y) # Oblicz loss
    optimizer.zero_grad()       # Zeruj gradienty (WAŻNE!)
    loss.backward()             # Backpropagation
    optimizer.step()            # Aktualizuj parametry
```

**Dlaczego `zero_grad()`?** PyTorch **akumuluje** gradienty (dodaje do istniejących). Bez zerowania gradzient rośnie z każdą iteracją!

In [ ]:
# Tensory PyTorch
a = torch.tensor([1.0, 2.0, 3.0])
print(a, a.dtype, a.shape)

# Konwersja NumPy ↔ PyTorch
numpy_arr = np.array([1, 2, 3], dtype=np.float32)
tensor = torch.from_numpy(numpy_arr)
back_to_numpy = tensor.numpy()

# Operacje (identyczne jak w NumPy)
b = torch.tensor([4.0, 5.0, 6.0])
print(a + b)       # elementowe
print(a @ b)       # dot product
print(a.mean())    # średnia

In [ ]:
# Autograd — automatyczne różniczkowanie!
x = torch.tensor(3.0, requires_grad=True)  # śledź operacje
y = x ** 2 + 2 * x + 1  # y = x² + 2x + 1

y.backward()  # oblicz gradient: dy/dx = 2x + 2
print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"dy/dx = {x.grad.item()}")  # 2*3 + 2 = 8 ✓

## 6. Regresja liniowa w PyTorch

In [ ]:
# Dane jako tensory
X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (100, 1)
y_t = torch.tensor(y_true, dtype=torch.float32).unsqueeze(1)  # (100, 1)

# Model liniowy: nn.Linear(in_features, out_features)
model = nn.Linear(1, 1)  # 1 wejście → 1 wyjście

# Funkcja kosztu i optymalizator
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Trening!
losses = []
for epoch in range(200):
    # Forward pass
    y_pred = model(X_t)
    loss = criterion(y_pred, y_t)
    
    # Backward pass + update
    optimizer.zero_grad()  # wyzeruj gradienty!
    loss.backward()        # oblicz gradienty
    optimizer.step()       # aktualizuj parametry
    
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")

# Wynik
w_pt = model.weight.item()
b_pt = model.bias.item()
print(f"\nPyTorch: w = {w_pt:.3f}, b = {b_pt:.3f}")

# Krzywa uczenia
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Krzywa uczenia — PyTorch")
plt.show()

---
---
# 🏋️ ZADANIA

---

### Zadanie 1: Wpływ learning rate (łatwe)

Uruchom gradient descent z NumPy (jak w sekcji 4) dla learning rate:
- 0.001 (za mały?)
- 0.01 (bazowy)
- 0.1 (za duży?)

Narysuj 3 krzywe uczenia na jednym wykresie. Co obserwujesz?

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: Regresja wielomianowa (średnie)

Dane: $y = 2x^2 - 3x + 5 + \text{szum}$

1. Wygeneruj dane i narysuj (scatter)
2. Dopasuj regresję liniową — jaki MSE?
3. Dodaj cechy wielomianowe ($x^2$) i dopasuj ponownie
4. Porównaj oba dopasowania na wykresie

In [ ]:
# Zadanie 2
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 3: Mini-batch Gradient Descent (trudne)

Zaimplementuj **mini-batch** gradient descent w NumPy:
- W każdej epoce losowo tasuj dane
- Dziel na mini-batche o rozmiarze 16
- Aktualizuj parametry po każdym batchu

Porównaj krzywe uczenia: full-batch vs mini-batch (na jednym wykresie).

In [ ]:
# Zadanie 3: Mini-batch GD
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 4: Regresja liniowa wielowymiarowa w PyTorch (trudne)

Wygeneruj dane 3D: $y = 2x_1 - x_2 + 0.5x_3 + 3 + \text{szum}$

1. Użyj `nn.Linear(3, 1)` 
2. Trenuj z `Adam` optimizer (zamiast SGD)
3. Narysuj krzywą uczenia
4. Porównaj odzyskane wagi z prawdziwymi

In [ ]:
# Zadanie 4: Regresja wielowymiarowa PyTorch
torch.manual_seed(42)
# TWÓJ KOD TUTAJ

---

## 🏆 Appendix OAI — Optymalizacja i gradient w kontekście olimpiady

Spadek gradientowy to **fundament** uczenia głębokiego. Na olimpiadzie spotkasz go w każdym zadaniu z PyTorch.

### Powiązane zadania OAI:
- **I OAI**: Adversarial attacks (FGSM = gradient + epsilon)
- **II OAI**: Inpainting z INR (optymalizacja sieci neuronowej)
- **III OAI**: Segmentacja wielospektralna (trening modelu z IoU)

### Ćwiczenia:
- **OAI-7A**: Implementacja schedulera learning rate (Cosine Annealing)
- **OAI-7B**: Gradient clipping — zapobieganie exploding gradients
- **OAI-7C**: Porównanie optymalizatorów (SGD vs Adam vs AdamW)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# === OAI-7A: Cosine Annealing Scheduler ===
print("=== Cosine Annealing LR Scheduler ===")

model_demo = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model_demo.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

lrs = []
for epoch in range(50):
    lrs.append(optimizer.param_groups[0]['lr'])
    optimizer.step()
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs, 'b-', linewidth=2)
plt.xlabel("Epoka")
plt.ylabel("Learning Rate")
plt.title("Cosine Annealing — używany w wielu rozwiązaniach OAI")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"  Start LR: {lrs[0]:.6f}")
print(f"  Min LR:   {min(lrs):.6f}")
print(f"  💡 Cosine Annealing pozwala modelowi 'doprecyzować' się pod koniec treningu")

# === OAI-7B: Gradient Clipping ===
print("\n=== Gradient Clipping ===")

# Symulacja modelu z exploding gradients
model_clip = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 1))
x = torch.randn(32, 10) * 10  # Duże wejścia
y = torch.randn(32, 1)

optimizer_clip = torch.optim.SGD(model_clip.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Bez clippingu
pred = model_clip(x)
loss = loss_fn(pred, y)
loss.backward()

grad_norm_before = torch.nn.utils.clip_grad_norm_(model_clip.parameters(), max_norm=float('inf'))
print(f"  Norma gradientu BEZ clippingu: {grad_norm_before:.4f}")

# Z clippingiem
optimizer_clip.zero_grad()
pred = model_clip(x)
loss = loss_fn(pred, y)
loss.backward()

grad_norm_after = torch.nn.utils.clip_grad_norm_(model_clip.parameters(), max_norm=1.0)
print(f"  Norma gradientu PO clippingu:  {grad_norm_after:.4f}")
print(f"  💡 Gradient clipping jest KLUCZOWY przy treningu RNN/LSTM")

# === OAI-7C: Porównanie optymalizatorów ===
print("\n=== Porównanie optymalizatorów ===")

def train_with_optimizer(opt_class, opt_kwargs, n_steps=200):
    """Trenuj prosty model i zwróć historię loss."""
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    X = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
    y = torch.tensor([[3.0], [7.0], [11.0], [15.0]])
    
    optimizer = opt_class(model.parameters(), **opt_kwargs)
    losses = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        loss = nn.MSELoss()(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

optimizers = {
    "SGD (lr=0.01)": (torch.optim.SGD, {"lr": 0.01}),
    "SGD+Momentum": (torch.optim.SGD, {"lr": 0.01, "momentum": 0.9}),
    "Adam (lr=0.01)": (torch.optim.Adam, {"lr": 0.01}),
    "AdamW (lr=0.01)": (torch.optim.AdamW, {"lr": 0.01, "weight_decay": 0.01}),
}

plt.figure(figsize=(10, 5))
for name, (opt_cls, kwargs) in optimizers.items():
    losses = train_with_optimizer(opt_cls, kwargs)
    plt.plot(losses, label=name, linewidth=2)

plt.xlabel("Krok")
plt.ylabel("Loss")
plt.title("Porównanie optymalizatorów — wiedza potrzebna na OAI")
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Na olimpiadzie: Adam/AdamW to bezpieczny domyślny wybór")
print("   SGD+Momentum lepszy przy dłuższym treningu (więcej epok)")